# Smart University Assistant — MIT Public Data Collection Pipeline

**Module:** Data Collection Only

This notebook implements a production-grade, scalable **web crawling and data collection pipeline** for MIT's publicly accessible websites. It discovers pages, extracts structured content, downloads PDFs, Office documents (.docx, .pptx, .xlsx), and announcement images (.png, .jpg), and organizes everything into a clean dataset ready to be consumed by downstream preprocessing / RAG stages.

**Scope — this notebook does:**
- Crawl a whitelisted set of MIT domains, staying strictly within them
- Respect `robots.txt` for every domain
- Extract clean, structured content from HTML pages (title, headings, tables, etc.)
- Discover and download PDF, Office (DOC/DOCX/PPT/XLS), and Image documents
- Persist per-page JSON documents and a master CSV index
- Provide checkpointing / resume support so a crawl can be interrupted and resumed
- Produce a final statistics report

## 2. Install Dependencies

We use `requests` + `BeautifulSoup` for static HTML, `tqdm` for progress bars, and `tenacity` for robust retry logic. Playwright is installed but only imported/used lazily if a page is detected to require JavaScript rendering.


In [ ]:
%%capture
!pip install -q requests beautifulsoup4 lxml tqdm tenacity playwright pandas
!playwright install --with-deps chromium


## 3. Imports

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import logging
import re
import sys
import time
import urllib.robotparser as robotparser
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin, urlparse, urlunparse, parse_qsl, urlencode

import requests
from bs4 import BeautifulSoup
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
from tqdm.auto import tqdm
import pandas as pd

print("All imports loaded successfully.")


All imports loaded successfully.


In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Project folder inside Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/SmartUniversityAssistant")

# Create the project directory if it doesn't exist
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/SmartUniversityAssistant


## 4. Configuration


In [ ]:
@dataclass(frozen=True)
class CrawlConfig:
    """Central configuration for the MIT data collection pipeline."""

    # Seed URLs the crawler starts from.
    seed_urls: tuple[str, ...] = (
        "https://catalog.mit.edu/",
        "https://registrar.mit.edu/",
        "https://oge.mit.edu/",
        "https://studentlife.mit.edu/",
        "https://libraries.mit.edu/",
    )

    # Only these domains (and subdomains) may be crawled.
    allowed_domains: tuple[str, ...] = (
        "catalog.mit.edu",
        "registrar.mit.edu",
        "oge.mit.edu",
        "studentlife.mit.edu",
        "libraries.mit.edu",
    )

    # Networking
    request_timeout_seconds: int = 30
    request_delay_seconds: float = 1.0
    max_retries: int = 3
    user_agent: str = (
        "SmartUniversityAssistantBot/1.0 "
        "(+educational-research-project; contact: data-collection@example.edu)"
    )

    # Crawl limits
    max_crawl_depth: int = 5
    max_pages: int = 1000
    max_pdfs: int = 500
    max_office_docs: int = 200
    max_images: int = 200

    # Behavior toggles
    respect_robots_txt: bool = True
    use_playwright_fallback: bool = False

    # Output paths
    project_root: Path = field(default_factory=lambda: PROJECT_ROOT)

    raw_html_dirname: str = "raw_html"
    html_text_dirname: str = "html"
    pdf_dirname: str = "pdfs"
    office_dirname: str = "office"
    images_dirname: str = "images"
    tables_dirname: str = "tables"
    metadata_dirname: str = "metadata"
    manifests_dirname: str = "manifests"
    logs_dirname: str = "logs"
    checkpoints_dirname: str = "checkpoints"
    json_dirname: str = "json"

    # Configurable file extensions
    pdf_extensions: tuple[str, ...] = ("PDF",)
    office_doc_extensions: tuple[str, ...] = ("DOC", "DOCX")
    office_ppt_extensions: tuple[str, ...] = ("PPT", "PPTX")
    office_xls_extensions: tuple[str, ...] = ("XLS", "XLSX")
    table_extensions: tuple[str, ...] = ("CSV",)
    text_extensions: tuple[str, ...] = ("TXT",)
    image_extensions: tuple[str, ...] = ("JPG", "JPEG", "PNG", "WEBP", "GIF")

    @property
    def office_extensions(self) -> tuple[str, ...]:
        return self.office_doc_extensions + self.office_ppt_extensions + self.office_xls_extensions

    @property
    def all_known_extensions(self) -> tuple[str, ...]:
        return (
            self.pdf_extensions
            + self.office_extensions
            + self.table_extensions
            + self.text_extensions
            + self.image_extensions
        )

    url_priority_keywords: tuple[str, ...] = (
        "registrar", "academic", "calendar", "course", "catalog",
        "student", "department", "faculty", "admissions", "handbook",
        "policy", "schedule", "housing", "research", "library",
    )

    @property
    def data_root(self) -> Path:
        return self.project_root / "data" / "raw"

    @property
    def raw_html_dir(self) -> Path:
        return self.data_root / self.raw_html_dirname

    @property
    def html_text_dir(self) -> Path:
        return self.data_root / self.html_text_dirname

    @property
    def pdf_dir(self) -> Path:
        return self.data_root / self.pdf_dirname

    @property
    def office_dir(self) -> Path:
        return self.data_root / self.office_dirname

    @property
    def images_dir(self) -> Path:
        return self.data_root / self.images_dirname

    @property
    def tables_dir(self) -> Path:
        return self.data_root / self.tables_dirname

    @property
    def metadata_dir(self) -> Path:
        return self.data_root / self.metadata_dirname

    @property
    def manifests_dir(self) -> Path:
        return self.data_root / self.manifests_dirname

    @property
    def logs_dir(self) -> Path:
        return self.data_root / self.logs_dirname

    @property
    def checkpoints_dir(self) -> Path:
        return self.data_root / self.checkpoints_dirname

    @property
    def json_dir(self) -> Path:
        return self.data_root / self.json_dirname

    @property
    def html_dir(self) -> Path:
        return self.html_text_dir

    def all_dirs(self) -> list[Path]:
        return [
            self.raw_html_dir,
            self.html_text_dir,
            self.pdf_dir,
            self.office_dir,
            self.images_dir,
            self.tables_dir,
            self.metadata_dir,
            self.manifests_dir,
            self.logs_dir,
            self.checkpoints_dir,
            self.json_dir,
        ]


CONFIG = CrawlConfig()

for directory in CONFIG.all_dirs():
    directory.mkdir(parents=True, exist_ok=True)

print("Configuration initialized. Output directories created at:")
for d in CONFIG.all_dirs():
    print(f"  - {d.resolve()}")


Configuration initialized. Output directories created at:
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/raw_html
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/html
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/pdfs
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/office
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/images
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/tables
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/metadata
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/manifests
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/logs
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/checkpoints
  - /content/drive/MyDrive/SmartUniversityAssistant/data/raw/json


## 5. Logging

In [ ]:
def setup_logger(config: CrawlConfig) -> logging.Logger:
    """Configure and return the project-wide logger."""
    logger = logging.getLogger("mit_crawler")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    log_path = config.metadata_dir / "crawl.log"
    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    return logger


logger = setup_logger(CONFIG)
logger.info("Logger initialized. Writing detailed logs to %s", CONFIG.metadata_dir / "crawl.log")


2026-07-27 19:41:04 | INFO     | mit_crawler | Logger initialized. Writing detailed logs to /content/drive/MyDrive/SmartUniversityAssistant/data/raw/metadata/crawl.log


INFO:mit_crawler:Logger initialized. Writing detailed logs to /content/drive/MyDrive/SmartUniversityAssistant/data/raw/metadata/crawl.log


## 6. Helper Functions

In [ ]:
def normalize_url(url: str) -> str:
    """Normalize a URL for consistent de-duplication, removing tracking query parameters."""
    parsed = urlparse(url)
    path = parsed.path.rstrip("/") or "/"

    # Remove common tracking parameters to prevent duplicate crawling
    tracking_params = {
        "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
        "fbclid", "gclid", "ref", "session", "sessionid"
    }

    query_params = parse_qsl(parsed.query, keep_blank_values=True)
    filtered_query = [(k, v) for k, v in query_params if k.lower() not in tracking_params]
    new_query = urlencode(filtered_query)

    normalized = urlunparse((
        parsed.scheme.lower(),
        parsed.netloc.lower(),
        path,
        "",
        new_query,
        "",
    ))
    return normalized


def get_domain(url: str) -> str:
    """Return the network location (domain) of a URL."""
    return urlparse(url).netloc.lower()


def is_allowed_domain(url: str, config: CrawlConfig) -> bool:
    """Check whether a URL's domain is within the whitelist."""
    domain = get_domain(url)
    return any(
        domain == allowed or domain.endswith("." + allowed)
        for allowed in config.allowed_domains
    )


def is_pdf_url(url: str) -> bool:
    """Heuristically detect whether a URL points to a PDF document."""
    return urlparse(url).path.lower().endswith(".pdf")


def is_office_url(url: str, config: CrawlConfig = CONFIG) -> bool:
    """Check if a URL points to an Office document (DOCX, PPTX, XLSX, etc.)."""
    ext = Path(urlparse(url).path).suffix.lower().lstrip(".")
    return ext.upper() in config.office_extensions


def is_image_url(url: str, config: CrawlConfig = CONFIG) -> bool:
    """Check if a URL points to an image file (PNG, JPG, WEBP, GIF, etc.)."""
    ext = Path(urlparse(url).path).suffix.lower().lstrip(".")
    return ext.upper() in config.image_extensions


def compute_document_id(url: str) -> str:
    """Generate a stable, short, deterministic ID for a URL."""
    return hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]


def infer_category(url: str) -> str:
    """Infer a coarse content category from the URL path/domain."""
    path = urlparse(url).path.lower()
    domain = get_domain(url)

    keyword_category_map = {
        "calendar": "academic_calendar",
        "catalog": "course_catalog",
        "subject": "subject_description",
        "course": "course_catalog",
        "degree": "degree_requirements",
        "requirement": "degree_requirements",
        "regulation": "student_regulations",
        "polic": "academic_policies",
        "handbook": "student_handbook",
        "department": "department_information",
        "admission": "admissions_information",
        "graduat": "graduation_requirements",
        "internship": "internship_information",
        "service": "student_services",
        "resource": "campus_resources",
        "faq": "faq",
        "news": "public_announcements",
        "announce": "public_announcements",
    }

    for keyword, category in keyword_category_map.items():
        if keyword in path:
            return category

    domain_category_map = {
        "catalog.mit.edu": "course_catalog",
        "registrar.mit.edu": "academic_records",
        "oge.mit.edu": "graduate_education",
        "studentlife.mit.edu": "student_services",
        "libraries.mit.edu": "campus_resources",
    }
    return domain_category_map.get(domain, "general")


def utc_now_iso() -> str:
    """Current UTC timestamp in ISO-8601 format."""
    return datetime.now(timezone.utc).isoformat()


def safe_filename(url: str, extension: str) -> str:
    """Build a filesystem-safe filename derived from a URL hash."""
    doc_id = compute_document_id(url)
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", urlparse(url).path).strip("_")[:60] or "index"
    return f"{doc_id}__{slug}.{extension}"


print("Helper functions defined (including URL normalization).")


Helper functions defined (including URL normalization).


## 6b. URL Priority Scoring & File-Type Classification

In [ ]:
def score_url_priority(url: str, config: CrawlConfig) -> int:
    haystack = url.lower()
    return sum(1 for keyword in config.url_priority_keywords if keyword in haystack)


def sort_by_priority(urls: list[str], config: CrawlConfig) -> list[str]:
    return sorted(urls, key=lambda u: score_url_priority(u, config), reverse=True)


def _extract_extension(url_or_name: str) -> str:
    path = urlparse(url_or_name).path or url_or_name
    suffix = Path(path).suffix.lower().lstrip(".")
    return suffix


def detect_file_type(url_or_name: str, config: CrawlConfig) -> tuple[str, str]:
    ext = _extract_extension(url_or_name)
    if not ext:
        return "unknown", ""

    ext_upper = ext.upper()
    if ext_upper in config.pdf_extensions:
        return "pdf", ext
    if ext_upper in config.office_extensions:
        return "office", ext
    if ext_upper in config.image_extensions:
        return "image", ext
    if ext_upper in config.table_extensions:
        return "table", ext
    if ext_upper in config.text_extensions:
        return "text", ext
    return "unknown", ext


def classify_page_type(url: str, config: CrawlConfig) -> str:
    family, _ext = detect_file_type(url, config)
    return family if family != "unknown" else "html"


print("Priority/classification helpers defined.")


Priority/classification helpers defined.


## 7. URL Manager

`URLManager` owns the crawl frontier, seen sets, domain robots.txt parsers, and specific target sets for PDFs, Office files, and Images.

In [ ]:
class URLManager:
    """Manages the crawl frontier, deduplication, and robots.txt compliance."""

    def __init__(self, config: CrawlConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.queue: list[tuple[str, int]] = []   # (url, depth)
        self.seen: set[str] = set()
        self.visited: set[str] = set()
        self.failed: dict[str, str] = {}
        self.pdf_urls: set[str] = set()
        self.office_urls: set[str] = set()
        self.image_urls: set[str] = set()
        self._robots_cache: dict[str, robotparser.RobotFileParser] = {}

        self.checkpoint_path = config.metadata_dir / "checkpoint.json"

    def seed(self) -> None:
        for url in self.config.seed_urls:
            self.add_url(url, depth=0)

    def add_url(self, url: str, depth: int) -> bool:
        normalized = normalize_url(url)

        if normalized in self.seen:
            return False
        if not is_allowed_domain(normalized, self.config):
            return False
        if depth > self.config.max_crawl_depth:
            return False

        self.seen.add(normalized)

        # Route binary / asset URLs directly to their download targets
        if is_pdf_url(normalized):
            self.pdf_urls.add(normalized)
            return True
        if is_office_url(normalized, self.config):
            self.office_urls.add(normalized)
            return True
        if is_image_url(normalized, self.config):
            self.image_urls.add(normalized)
            return True

        if not self.robots_allows(normalized):
            self.logger.info("Skipping (robots.txt disallows): %s", normalized)
            return False

        self.queue.append((normalized, depth))
        # Prioritize URLs based on score. Higher score first, then shallower depths.
        self.queue.sort(key=lambda x: (score_url_priority(x[0], self.config), -x[1]), reverse=True)
        return True

    def next_url(self) -> Optional[tuple[str, int]]:
        if not self.queue:
            return None
        return self.queue.pop(0)

    def has_capacity(self) -> bool:
        return len(self.visited) < self.config.max_pages

    def robots_allows(self, url: str) -> bool:
        if not self.config.respect_robots_txt:
            return True

        parsed = urlparse(url)
        domain_key = f"{parsed.scheme}://{parsed.netloc}"

        if domain_key not in self._robots_cache:
            rp = robotparser.RobotFileParser()
            rp.set_url(urljoin(domain_key, "/robots.txt"))
            try:
                rp.read()
            except Exception as exc:  # noqa: BLE001
                self.logger.warning("Could not fetch robots.txt for %s (%s); defaulting to allow.", domain_key, exc)
                self._robots_cache[domain_key] = None  # type: ignore[assignment]
                return True
            self._robots_cache[domain_key] = rp

        rp = self._robots_cache[domain_key]
        if rp is None:
            return True
        return rp.can_fetch(self.config.user_agent, url)

    def save_checkpoint(self) -> None:
        state = {
            "queue": self.queue,
            "seen": list(self.seen),
            "visited": list(self.visited),
            "failed": self.failed,
            "pdf_urls": list(self.pdf_urls),
            "office_urls": list(self.office_urls),
            "image_urls": list(self.image_urls),
            "saved_at": utc_now_iso(),
        }
        with open(self.checkpoint_path, "w", encoding="utf-8") as f:
            json.dump(state, f, indent=2)

    def load_checkpoint(self) -> bool:
        if not self.checkpoint_path.exists():
            return False
        with open(self.checkpoint_path, "r", encoding="utf-8") as f:
            state = json.load(f)
        self.queue = [tuple(item) for item in state.get("queue", [])]
        self.seen = set(state.get("seen", []))
        self.visited = set(state.get("visited", []))
        self.failed = state.get("failed", {})
        self.pdf_urls = set(state.get("pdf_urls", []))
        self.office_urls = set(state.get("office_urls", []))
        self.image_urls = set(state.get("image_urls", []))
        self.logger.info(
            "Resumed from checkpoint: %d queued, %d visited, %d pdfs, %d office, %d images pending.",
            len(self.queue), len(self.visited), len(self.pdf_urls), len(self.office_urls), len(self.image_urls)
        )
        return True


print("URLManager updated with queue sorting for priority urls.")


URLManager updated with queue sorting for priority urls.


## 8. HTTP Fetcher

In [ ]:
@dataclass
class FetchResult:
    url: str
    final_url: str
    status_code: Optional[int]
    content: Optional[bytes]
    content_type: str
    elapsed_seconds: float
    error: Optional[str] = None

    @property
    def ok(self) -> bool:
        return (
            self.error is None
            and self.status_code is not None
            and 200 <= self.status_code < 300
        )


def build_session(config: CrawlConfig) -> requests.Session:
    session = requests.Session()
    session.headers.update({
        "User-Agent": config.user_agent,
        "Accept": "text/html,application/pdf,application/xhtml+xml,*/*;q=0.8",
    })
    return session


class RetryableFetchError(Exception):
    """Raised for transient errors worth retrying."""


@retry(
    reraise=True,
    stop=stop_after_attempt(CONFIG.max_retries + 1),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type(RetryableFetchError),
)
def _get_with_retry(session: requests.Session, url: str, config: CrawlConfig) -> requests.Response:
    try:
        response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
    except (requests.ConnectionError, requests.Timeout) as exc:
        raise RetryableFetchError(str(exc)) from exc

    if response.status_code in (429, 500, 502, 503, 504):
        raise RetryableFetchError(f"HTTP {response.status_code}")

    return response


def fetch_url(
    url: str,
    config: CrawlConfig,
    session: requests.Session,
    logger: logging.Logger,
) -> FetchResult:
    start = time.monotonic()
    try:
        response = _get_with_retry(session, url, config)
        elapsed = time.monotonic() - start
        result = FetchResult(
            url=url,
            final_url=response.url,
            status_code=response.status_code,
            content=response.content if response.status_code == 200 else None,
            content_type=response.headers.get("Content-Type", ""),
            elapsed_seconds=elapsed,
        )
        if response.status_code != 200:
            result.error = f"HTTP {response.status_code}"
            logger.warning("Non-200 response for %s: %s", url, result.error)
        return result
    except Exception as exc:  # noqa: BLE001
        elapsed = time.monotonic() - start
        logger.error("Failed to fetch %s after retries: %s", url, exc)
        return FetchResult(
            url=url,
            final_url=url,
            status_code=None,
            content=None,
            content_type="",
            elapsed_seconds=elapsed,
            error=str(exc),
        )
    finally:
        time.sleep(config.request_delay_seconds)


print("Fetcher defined.")


Fetcher defined.


## 9. HTML Content Extraction

Extracts structured page content and routes outlinks into HTML, PDF, Office document, and Image buckets.

In [ ]:
BOILERPLATE_TAGS = ("script", "style", "noscript", "svg", "iframe", "form")
BOILERPLATE_SELECTORS = (
    "nav", "footer", "header", "aside",
    "[role='navigation']", ".skip-link", ".site-footer", ".site-header",
    # Breadcrumbs
    ".breadcrumb", "#breadcrumbs",
    # Cookie banners
    "#cookie-banner", ".cookie-notice", "#cookie-law-info-bar",
    # Accessibility widgets
    "#accessibility-menu", ".accessibility-widget",
    # Sidebars
    ".sidebar", "#sidebar",
    # Search boxes
    "form[role='search']", ".search-box", "#search-form",
    # Social sharing widgets
    ".social-share", ".share-buttons",
    # Announcement banners
    ".announcement", ".alert", ".site-notice",
)


@dataclass
class ExtractedPage:
    """Structured representation of a single crawled HTML page."""

    doc_id: str
    url: str
    domain: str
    category: str
    title: str
    headings: list[dict]
    tables: list[list[list[str]]]
    paragraphs: list[str]
    full_text: str
    word_count: int
    outlinks: list[str]
    pdf_links: list[str]
    office_links: list[str]
    image_links: list[str]
    content_hash: str
    fetched_at: str
    status_code: Optional[int]
    content_type: str


def _clean_soup(soup: BeautifulSoup) -> None:
    for tag_name in BOILERPLATE_TAGS:
        for tag in soup.find_all(tag_name):
            tag.decompose()
    for selector in BOILERPLATE_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()


def _extract_headings(soup: BeautifulSoup) -> list[dict]:
    headings = []
    for level in range(1, 7):
        for tag in soup.find_all(f"h{level}"):
            text = tag.get_text(strip=True)
            if text:
                headings.append({"level": level, "text": text})
    return headings


def _extract_tables(soup: BeautifulSoup) -> list[list[list[str]]]:
    tables = []
    for table_tag in soup.find_all("table"):
        rows = []
        for row_tag in table_tag.find_all("tr"):
            cells = [c.get_text(strip=True) for c in row_tag.find_all(["td", "th"])]
            if cells:
                rows.append(cells)
        if rows:
            tables.append(rows)
    return tables


def _extract_paragraphs(soup: BeautifulSoup) -> list[str]:
    paragraphs = []
    for p_tag in soup.find_all(["p", "li"]):
        text = p_tag.get_text(strip=True)
        if text and len(text) > 1:
            paragraphs.append(text)
    return paragraphs


def _extract_links(soup: BeautifulSoup, base_url: str, config: CrawlConfig) -> tuple[list[str], list[str], list[str], list[str]]:
    """Return (page_links, pdf_links, office_links, image_links) discovered on the page."""
    page_links: list[str] = []
    pdf_links: list[str] = []
    office_links: list[str] = []
    image_links: list[str] = []

    # Check all anchor tags
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"].strip()
        if not href or href.startswith(("mailto:", "tel:", "javascript:")):
            continue
        absolute = normalize_url(urljoin(base_url, href))
        if not is_allowed_domain(absolute, config):
            continue

        if is_pdf_url(absolute):
            pdf_links.append(absolute)
        elif is_office_url(absolute, config):
            office_links.append(absolute)
        elif is_image_url(absolute, config):
            image_links.append(absolute)
        else:
            page_links.append(absolute)

    # Check image tags directly for announcements/embedded photos
    for img_tag in soup.find_all("img", src=True):
        src = img_tag["src"].strip()
        if not src or src.startswith("data:"):
            continue
        absolute = normalize_url(urljoin(base_url, src))
        if is_allowed_domain(absolute, config) and is_image_url(absolute, config):
            image_links.append(absolute)

    return page_links, pdf_links, office_links, image_links


def extract_html_content(
    url: str,
    html_bytes: bytes,
    config: CrawlConfig,
    status_code: Optional[int],
    content_type: str,
) -> ExtractedPage:
    soup = BeautifulSoup(html_bytes, "lxml")
    _clean_soup(soup)

    title_tag = soup.find("title")
    title = title_tag.get_text(strip=True) if title_tag else urlparse(url).path

    headings = _extract_headings(soup)
    tables = _extract_tables(soup)
    paragraphs = _extract_paragraphs(soup)
    page_links, pdf_links, office_links, image_links = _extract_links(soup, url, config)

    full_text = "\n".join(paragraphs) if paragraphs else soup.get_text(separator="\n", strip=True)
    word_count = len(full_text.split())
    content_hash = hashlib.sha256(full_text.encode("utf-8")).hexdigest()

    return ExtractedPage(
        doc_id=compute_document_id(url),
        url=url,
        domain=get_domain(url),
        category=infer_category(url),
        title=title,
        headings=headings,
        tables=tables,
        paragraphs=paragraphs,
        full_text=full_text,
        word_count=word_count,
        outlinks=page_links,
        pdf_links=pdf_links,
        office_links=office_links,
        image_links=image_links,
        content_hash=content_hash,
        fetched_at=utc_now_iso(),
        status_code=status_code,
        content_type=content_type,
    )


print("HTML extraction updated: stronger boilerplate removal.")


HTML extraction updated: stronger boilerplate removal.


## 10. File Downloaders (PDF, Office, and Images)

Generic binary download logic that handles PDFs, Word/Excel/PowerPoint documents, and announcement images, validating MIME types to prevent downloading invalid files.

In [ ]:
@dataclass
class FileDownloadResult:
    """Metadata describing a downloaded (or failed) binary asset."""

    doc_id: str
    url: str
    domain: str
    category: str
    file_type: str        # 'pdf', 'office', 'image'
    file_path: Optional[str]
    file_size_bytes: Optional[int]
    content_hash: Optional[str]
    downloaded_at: str
    status_code: Optional[int]
    error: Optional[str] = None


PDFDownloadResult = FileDownloadResult


def download_binary_file(
    url: str,
    target_dir: Path,
    file_type_label: str,
    config: CrawlConfig,
    session: requests.Session,
    logger: logging.Logger,
) -> FileDownloadResult:
    """Download a binary asset, validate its MIME type, and persist it to target_dir."""
    fetch_result = fetch_url(url, config, session, logger)

    if not fetch_result.ok or fetch_result.content is None:
        return FileDownloadResult(
            doc_id=compute_document_id(url),
            url=url,
            domain=get_domain(url),
            category=infer_category(url),
            file_type=file_type_label,
            file_path=None,
            file_size_bytes=None,
            content_hash=None,
            downloaded_at=utc_now_iso(),
            status_code=fetch_result.status_code,
            error=fetch_result.error or "empty content",
        )

    # Validate MIME type
    content_type = fetch_result.content_type.lower()
    is_valid_mime = False

    if file_type_label == "pdf" and "application/pdf" in content_type:
        is_valid_mime = True
    elif file_type_label == "office" and any(mime in content_type for mime in [
        "application/vnd.", "application/msword", "application/msexcel", "application/mspowerpoint"
    ]):
        is_valid_mime = True
    elif file_type_label == "image" and "image/" in content_type:
        is_valid_mime = True

    if not is_valid_mime:
        return FileDownloadResult(
            doc_id=compute_document_id(url),
            url=url,
            domain=get_domain(url),
            category=infer_category(url),
            file_type=file_type_label,
            file_path=None,
            file_size_bytes=None,
            content_hash=None,
            downloaded_at=utc_now_iso(),
            status_code=fetch_result.status_code,
            error=f"Invalid content type for {file_type_label}: {content_type}",
        )

    ext = Path(urlparse(url).path).suffix.lower().lstrip(".") or file_type_label
    filename = safe_filename(url, ext)
    file_path = target_dir / filename
    with open(file_path, "wb") as f:
        f.write(fetch_result.content)

    content_hash = hashlib.sha256(fetch_result.content).hexdigest()

    return FileDownloadResult(
        doc_id=compute_document_id(url),
        url=url,
        domain=get_domain(url),
        category=infer_category(url),
        file_type=file_type_label,
        file_path=str(file_path),
        file_size_bytes=len(fetch_result.content),
        content_hash=content_hash,
        downloaded_at=utc_now_iso(),
        status_code=fetch_result.status_code,
    )


def download_pdf(url: str, config: CrawlConfig, session: requests.Session, logger: logging.Logger) -> FileDownloadResult:
    return download_binary_file(url, config.pdf_dir, "pdf", config, session, logger)


def download_office_doc(url: str, config: CrawlConfig, session: requests.Session, logger: logging.Logger) -> FileDownloadResult:
    return download_binary_file(url, config.office_dir, "office", config, session, logger)


def download_image(url: str, config: CrawlConfig, session: requests.Session, logger: logging.Logger) -> FileDownloadResult:
    return download_binary_file(url, config.images_dir, "image", config, session, logger)


print("Downloaders updated: Added strict MIME type validation.")


Downloaders updated: Added strict MIME type validation.


## 11. Playwright Fallback (JS-Rendered Pages)

In [ ]:
import concurrent.futures

JS_RENDER_MARKERS = (
    "enable javascript",
    "please enable js",
    "you need to enable javascript",
    "requires javascript",
)

MIN_WORDS_BEFORE_JS_FALLBACK = 40


def needs_js_rendering(extracted: ExtractedPage, raw_html: bytes) -> bool:
    if extracted.word_count >= MIN_WORDS_BEFORE_JS_FALLBACK:
        return False

    lowered_text = extracted.full_text.lower()
    if any(marker in lowered_text for marker in JS_RENDER_MARKERS):
        return True

    raw_html_lower = raw_html.lower()
    if any(marker.encode() in raw_html_lower for marker in JS_RENDER_MARKERS):
        return True

    if len(raw_html) > 20_000 and extracted.word_count < MIN_WORDS_BEFORE_JS_FALLBACK:
        return True

    return False


def fetch_with_playwright(url: str, config: CrawlConfig, logger: logging.Logger) -> Optional[bytes]:
    if not config.use_playwright_fallback:
        return None

    try:
        from playwright.sync_api import sync_playwright
    except ImportError:
        logger.warning("Playwright not available; skipping JS-rendering fallback for %s", url)
        return None

    def _render() -> Optional[bytes]:
        try:
            with sync_playwright() as p:
                browser = p.chromium.launch(headless=True)
                page = browser.new_page(user_agent=config.user_agent)
                page.goto(url, timeout=config.request_timeout_seconds * 1000, wait_until="networkidle")
                html = page.content()
                browser.close()
                return html.encode("utf-8")
        except Exception as exc:  # noqa: BLE001
            logger.warning("Playwright rendering failed for %s: %s", url, exc)
            return None

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(_render).result()


print("Playwright fallback defined.")


Playwright fallback defined.


## 12. Crawler Orchestrator

`MITCrawler` ties together URL discovery, page crawling, and download post-processing for PDFs, Office documents, and announcement images.

In [ ]:
CHECKPOINT_EVERY_N_PAGES = 10


class MITCrawler:
    """Top-level orchestrator for crawling and binary file downloading."""

    def __init__(self, config: CrawlConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.url_manager = URLManager(config, logger)
        self.session = build_session(config)
        self.stats = {
            "pages_crawled": 0,
            "pages_failed": 0,
            "pdfs_downloaded": 0,
            "pdfs_failed": 0,
            "office_downloaded": 0,
            "office_failed": 0,
            "images_downloaded": 0,
            "images_failed": 0,
            "js_fallback_used": 0,
            "started_at": None,
            "finished_at": None,
        }
        self.failed_urls_log: list[dict] = []

    def _save_page_json(self, page: ExtractedPage) -> Path:
        out_path = self.config.json_dir / f"{page.doc_id}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(asdict(page), f, indent=2, ensure_ascii=False)
        return out_path

    def _save_page_html_snapshot(self, page: ExtractedPage, raw_html: bytes) -> None:
        out_path = self.config.html_dir / f"{page.doc_id}.txt"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(f"URL: {page.url}\nTITLE: {page.title}\n\n{page.full_text}")

    def _save_asset_metadata(self, result: FileDownloadResult, log_filename: str) -> None:
        out_path = self.config.metadata_dir / log_filename
        with open(out_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(asdict(result), ensure_ascii=False) + "\n")

    def _process_page(self, url: str, depth: int) -> None:
        fetch_result = fetch_url(url, self.config, self.session, self.logger)

        if not fetch_result.ok or fetch_result.content is None:
            error_msg = fetch_result.error or "unknown error"
            self.url_manager.failed[url] = error_msg
            self.stats["pages_failed"] += 1
            self.failed_urls_log.append({
                "URL": url,
                "Crawl Stage": "Page Fetch",
                "HTTP Status Code": fetch_result.status_code,
                "Error Message": error_msg,
                "Timestamp": utc_now_iso()
            })
            return

        raw_html = fetch_result.content
        page = extract_html_content(
            url, raw_html, self.config, fetch_result.status_code, fetch_result.content_type
        )

        if needs_js_rendering(page, raw_html):
            self.logger.info("Triggering Playwright fallback for %s", url)
            rendered_html = fetch_with_playwright(url, self.config, self.logger)
            if rendered_html:
                page = extract_html_content(
                    url, rendered_html, self.config, fetch_result.status_code, "text/html (rendered)"
                )
                raw_html = rendered_html
                self.stats["js_fallback_used"] += 1

        self._save_page_json(page)
        self._save_page_html_snapshot(page, raw_html)

        self.url_manager.visited.add(url)
        self.stats["pages_crawled"] += 1

        for link in page.outlinks:
            self.url_manager.add_url(link, depth + 1)
        for pdf_link in page.pdf_links:
            self.url_manager.pdf_urls.add(normalize_url(pdf_link))
        for office_link in page.office_links:
            self.url_manager.office_urls.add(normalize_url(office_link))
        for img_link in page.image_links:
            self.url_manager.image_urls.add(normalize_url(img_link))

    def _process_pdfs(self) -> None:
        pending_pdfs = list(self.url_manager.pdf_urls - self.url_manager.visited)[: self.config.max_pdfs]
        for pdf_url in tqdm(pending_pdfs, desc="Downloading PDFs", unit="pdf"):
            result = download_pdf(pdf_url, self.config, self.session, self.logger)
            self._save_asset_metadata(result, "pdf_index.jsonl")
            self.url_manager.visited.add(pdf_url)
            if result.error:
                self.stats["pdfs_failed"] += 1
                self.failed_urls_log.append({
                    "URL": pdf_url,
                    "Crawl Stage": "PDF Download",
                    "HTTP Status Code": result.status_code,
                    "Error Message": result.error,
                    "Timestamp": result.downloaded_at
                })
            else:
                self.stats["pdfs_downloaded"] += 1

    def _process_office_docs(self) -> None:
        pending_office = list(self.url_manager.office_urls - self.url_manager.visited)[: self.config.max_office_docs]
        for office_url in tqdm(pending_office, desc="Downloading Office Docs", unit="doc"):
            result = download_office_doc(office_url, self.config, self.session, self.logger)
            self._save_asset_metadata(result, "office_index.jsonl")
            self.url_manager.visited.add(office_url)
            if result.error:
                self.stats["office_failed"] += 1
                self.failed_urls_log.append({
                    "URL": office_url,
                    "Crawl Stage": "Office Download",
                    "HTTP Status Code": result.status_code,
                    "Error Message": result.error,
                    "Timestamp": result.downloaded_at
                })
            else:
                self.stats["office_downloaded"] += 1

    def _process_images(self) -> None:
        pending_images = list(self.url_manager.image_urls - self.url_manager.visited)[: self.config.max_images]
        for img_url in tqdm(pending_images, desc="Downloading Images", unit="img"):
            result = download_image(img_url, self.config, self.session, self.logger)
            self._save_asset_metadata(result, "image_index.jsonl")
            self.url_manager.visited.add(img_url)
            if result.error:
                self.stats["images_failed"] += 1
                self.failed_urls_log.append({
                    "URL": img_url,
                    "Crawl Stage": "Image Download",
                    "HTTP Status Code": result.status_code,
                    "Error Message": result.error,
                    "Timestamp": result.downloaded_at
                })
            else:
                self.stats["images_downloaded"] += 1

    def run(self, resume: bool = False) -> dict:
        self.stats["started_at"] = utc_now_iso()

        loaded = resume and self.url_manager.load_checkpoint()
        if not loaded:
            self.url_manager.seed()

        self.logger.info("Starting crawl. Queue size: %d", len(self.url_manager.queue))

        progress = tqdm(
            total=self.config.max_pages,
            initial=len(self.url_manager.visited),
            desc="Crawling pages",
            unit="page",
        )
        try:
            while self.url_manager.has_capacity():
                next_item = self.url_manager.next_url()
                if next_item is None:
                    self.logger.info("Frontier exhausted; nothing left to crawl.")
                    break

                url, depth = next_item
                if url in self.url_manager.visited:
                    continue

                self._process_page(url, depth)
                progress.update(1)

                if self.stats["pages_crawled"] % CHECKPOINT_EVERY_N_PAGES == 0:
                    self.url_manager.save_checkpoint()
        except KeyboardInterrupt:
            self.logger.warning("Crawl interrupted by user. Saving checkpoint before exiting.")
        finally:
            progress.close()
            self.url_manager.save_checkpoint()

        self.logger.info("Page crawl finished. Starting PDF downloads.")
        self._process_pdfs()
        self.logger.info("Starting Office document downloads.")
        self._process_office_docs()
        self.logger.info("Starting image downloads.")
        self._process_images()
        self.url_manager.save_checkpoint()

        self.stats["finished_at"] = utc_now_iso()

        stats_path = self.config.metadata_dir / "run_stats.json"
        with open(stats_path, "w", encoding="utf-8") as f:
            json.dump(self.stats, f, indent=2)

        # --------------------------------------------------------
        # Generate Crawl Manifest (Improvement 5)
        # --------------------------------------------------------
        execution_time = (datetime.fromisoformat(self.stats["finished_at"]) - datetime.fromisoformat(self.stats["started_at"])).total_seconds()

        manifest = {
            "crawl_version": "1.0",
            "execution_timestamp": utc_now_iso(),
            "seed_urls": list(self.config.seed_urls),
            "allowed_domains": list(self.config.allowed_domains),
            "crawler_configuration": {
                "request_timeout_seconds": self.config.request_timeout_seconds,
                "request_delay_seconds": self.config.request_delay_seconds,
                "max_retries": self.config.max_retries,
                "max_crawl_depth": self.config.max_crawl_depth,
                "max_pages": self.config.max_pages,
                "max_pdfs": self.config.max_pdfs,
                "max_office_docs": self.config.max_office_docs,
                "max_images": self.config.max_images,
            },
            "statistics": {
                "total_pages_crawled": self.stats["pages_crawled"],
                "pdfs_downloaded": self.stats["pdfs_downloaded"],
                "office_documents_downloaded": self.stats["office_downloaded"],
                "images_downloaded": self.stats["images_downloaded"],
                "failed_pages": self.stats["pages_failed"],
                "failed_downloads": self.stats["pdfs_failed"] + self.stats["office_failed"] + self.stats["images_failed"],
                "execution_time_seconds": execution_time
            }
        }
        manifest_path = self.config.metadata_dir / "crawl_manifest.json"
        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)

        # --------------------------------------------------------
        # Generate Failed URL Report (Improvement 6)
        # --------------------------------------------------------
        failed_df = pd.DataFrame(self.failed_urls_log, columns=["URL", "Crawl Stage", "HTTP Status Code", "Error Message", "Timestamp"])
        failed_csv_path = self.config.metadata_dir / "failed_urls.csv"
        failed_df.to_csv(failed_csv_path, index=False)

        return self.stats


print("MITCrawler orchestrator updated with manifest and failure reporting.")


MITCrawler orchestrator updated with manifest and failure reporting.


## 13. Quick  Test

In [ ]:
from dataclasses import replace

SMOKE_TEST_CONFIG = replace(CONFIG, max_pages=10, max_pdfs=3, max_office_docs=3, max_images=3, max_crawl_depth=2)

smoke_logger = setup_logger(SMOKE_TEST_CONFIG)
smoke_crawler = MITCrawler(SMOKE_TEST_CONFIG, smoke_logger)
smoke_stats = smoke_crawler.run(resume=False)

print("\nSmoke test complete. Summary:")
for key, value in smoke_stats.items():
    print(f"  {key}: {value}")


2026-07-27 19:41:05 | INFO     | mit_crawler | Starting crawl. Queue size: 5


INFO:mit_crawler:Starting crawl. Queue size: 5


Crawling pages:   0%|          | 0/10 [00:00<?, ?page/s]

2026-07-27 19:41:19 | INFO     | mit_crawler | Page crawl finished. Starting PDF downloads.


INFO:mit_crawler:Page crawl finished. Starting PDF downloads.


2026-07-27 19:41:23 | INFO     | mit_crawler | Starting Office document downloads.


INFO:mit_crawler:Starting Office document downloads.


2026-07-27 19:41:23 | INFO     | mit_crawler | Starting image downloads.


INFO:mit_crawler:Starting image downloads.



Smoke test complete. Summary:
  pages_crawled: 10
  pages_failed: 0
  pdfs_downloaded: 3
  pdfs_failed: 0
  office_downloaded: 0
  office_failed: 0
  images_downloaded: 3
  images_failed: 0
  js_fallback_used: 0
  started_at: 2026-07-27T19:41:05.093666+00:00
  finished_at: 2026-07-27T19:41:26.676087+00:00


## 14. Run the Full Crawl

In [ ]:
crawler = MITCrawler(CONFIG, logger)
run_stats = crawler.run(resume=False)

print("\nCrawl run complete. Summary:")
for key, value in run_stats.items():
    print(f"  {key}: {value}")


2026-07-27 19:41:27 | INFO     | mit_crawler | Starting crawl. Queue size: 5


INFO:mit_crawler:Starting crawl. Queue size: 5


Crawling pages:   0%|          | 0/1000 [00:00<?, ?page/s]

2026-07-27 19:41:35 | INFO     | mit_crawler | Triggering Playwright fallback for https://registrar.mit.edu/calendar?f%5B0%5D=category%3A99&f%5B1%5D=student%3A92


INFO:mit_crawler:Triggering Playwright fallback for https://registrar.mit.edu/calendar?f%5B0%5D=category%3A99&f%5B1%5D=student%3A92


...

2026-07-27 20:20:53 | INFO     | mit_crawler | Starting Office document downloads.


INFO:mit_crawler:Starting Office document downloads.


2026-07-27 20:20:55 | INFO     | mit_crawler | Starting image downloads.


INFO:mit_crawler:Starting image downloads.



Crawl run complete. Summary:
  pages_crawled: 1000
  pages_failed: 17
  pdfs_downloaded: 141
  pdfs_failed: 231
  office_downloaded: 2
  office_failed: 0
  images_downloaded: 127
  images_failed: 0
  js_fallback_used: 0
  started_at: 2026-07-27T19:41:26.723378+00:00
  finished_at: 2026-07-27T20:23:14.074656+00:00


## 15. Build the Master CSV Index

Scans every per-page JSON document plus the PDF, Office, and Image metadata logs and flattens them into a single `master_index.csv` under `data/raw/metadata/`.

In [ ]:
def build_master_index(config: CrawlConfig) -> pd.DataFrame:
    rows = []

    for json_path in sorted(config.json_dir.glob("*.json")):
        with open(json_path, "r", encoding="utf-8") as f:
            doc = json.load(f)
        rows.append({
            "doc_id": doc["doc_id"],
            "doc_type": "html",
            "url": doc["url"],
            "domain": doc["domain"],
            "category": doc["category"],
            "title": doc["title"],
            "word_count": doc["word_count"],
            "num_headings": len(doc["headings"]),
            "num_tables": len(doc["tables"]),
            "num_outlinks": len(doc["outlinks"]),
            "content_hash": doc["content_hash"],
            "fetched_at": doc["fetched_at"],
            "status_code": doc["status_code"],
            "file_path": str(json_path),
        })

    asset_logs = [
        ("pdf_index.jsonl", "pdf"),
        ("office_index.jsonl", "office"),
        ("image_index.jsonl", "image"),
    ]

    for log_filename, doc_type in asset_logs:
        index_path = config.metadata_dir / log_filename
        if index_path.exists():
            with open(index_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    doc = json.loads(line)
                    if doc.get("error"):
                        continue
                    rows.append({
                        "doc_id": doc["doc_id"],
                        "doc_type": doc_type,
                        "url": doc["url"],
                        "domain": doc["domain"],
                        "category": doc["category"],
                        "title": Path(doc["file_path"]).name if doc["file_path"] else "",
                        "word_count": None,
                        "num_headings": None,
                        "num_tables": None,
                        "num_outlinks": None,
                        "content_hash": doc["content_hash"],
                        "fetched_at": doc["downloaded_at"],
                        "status_code": doc["status_code"],
                        "file_path": doc["file_path"],
                    })

    df = pd.DataFrame(rows)

    if not df.empty:
        before = len(df)
        df = df.drop_duplicates(subset=["content_hash"], keep="first")
        deduped = before - len(df)
        if deduped:
            print(f"Removed {deduped} duplicate document(s) by content hash.")

    index_path = config.metadata_dir / "master_index.csv"
    df.to_csv(index_path, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"Master index written to {index_path.resolve()} ({len(df)} rows).")
    return df


master_df = build_master_index(CONFIG)
master_df.head(10)


Removed 166 duplicate document(s) by content hash.
Master index written to /content/drive/MyDrive/SmartUniversityAssistant/data/raw/metadata/master_index.csv (1110 rows).


,doc_id,doc_type,url,domain,category,title,word_count,num_headings,num_tables,num_outlinks,content_hash,fetched_at,status_code,file_path
0,0029ae65943eda2f,html,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,registrar.mit.edu,academic_calendar,Academic Calendar | MIT Registrar,30.0,2.0,1.0,18.0,4b9164a149dd2775bfc68889627285e9119d60311a20ae...,2026-07-27T19:41:45.939483+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
1,0040472cc177908d,html,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,registrar.mit.edu,academic_calendar,Academic Calendar | MIT Registrar,168.0,8.0,7.0,24.0,51306fb1c9f34c63087687cc787be0cd8691607f333242...,2026-07-27T19:44:57.799851+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
2,0092e825b91cc23b,html,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,registrar.mit.edu,academic_calendar,Academic Calendar | MIT Registrar,309.0,14.0,13.0,30.0,e4d92803544c1f23fd1aa993105b2e75512c2ffcfac3e5...,2026-07-27T19:48:14.481432+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
3,00a804e29234cc32,html,https://studentlife.mit.edu/campus-communities...,studentlife.mit.edu,student_services,Parents and Families - MIT Division of Student...,1061.0,11.0,0.0,0.0,856d23b0a34bb4733cb37d9f9bf25f5f159810533e4e14...,2026-07-27T20:01:59.119811+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
4,00cf5a5ea75ca24a,html,https://studentlife.mit.edu/campus-communities...,studentlife.mit.edu,student_services,MIT’s Mason Estrada to sign with the Los Angel...,1063.0,6.0,0.0,4.0,410c0f1d7af281a2c572043cdea6f647c85e53be6c6230...,2026-07-27T20:10:29.688645+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
5,0144978e4092cdee,html,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,registrar.mit.edu,academic_calendar,Academic Calendar | MIT Registrar,248.0,14.0,13.0,30.0,3dfe6c3f966753db057d3e2e061e146c58f82cee6251db...,2026-07-27T19:45:00.203171+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
6,01a0cb5675163e39,html,https://registrar.mit.edu/faculty-curriculum-s...,registrar.mit.edu,academic_records,Compass Initiative | MIT Registrar,112.0,2.0,0.0,4.0,b1cabcd39bc44536c4a42c98190bdd44553983c4567151...,2026-07-27T20:03:23.328313+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
7,01b7695079742e61,html,https://registrar.mit.edu/faculty-curriculum-s...,registrar.mit.edu,academic_records,Hack Yourself: A New Data Science and CI-H Sub...,154.0,2.0,0.0,4.0,325e2643d611fbd3ae2b379d6fe7a1011d4614513fb120...,2026-07-27T20:03:32.323475+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
8,01bdd1ce4ca2bb51,html,https://catalog.mit.edu/summer/subjects,catalog.mit.edu,subject_description,Subjects | MIT Course Catalog,794.0,9.0,2.0,142.0,25d6d872007863afc9fdcad347885ea7d22986115c7891...,2026-07-27T19:56:35.997962+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...
9,020f973090024aeb,html,https://registrar.mit.edu/calendar?f%5B0%5D=ca...,registrar.mit.edu,academic_calendar,Academic Calendar | MIT Registrar,1630.0,14.0,13.0,30.0,ba55cbbafbad17a012ea7987c8f6f55a3f97aa7635f068...,2026-07-27T19:45:52.722296+00:00,200,/content/drive/MyDrive/SmartUniversityAssistan...


## 16. Final Statistics Report

In [ ]:
def print_final_report(
    config: CrawlConfig,
    url_manager: URLManager,
    stats: dict,
    master_df: pd.DataFrame,
) -> None:
    print("=" * 70)
    print("MIT DATA COLLECTION — FINAL REPORT")
    print("=" * 70)

    print(f"\nRun window: {stats.get('started_at')}  ->  {stats.get('finished_at')}")
    print(f"\nPages crawled successfully : {stats.get('pages_crawled', 0)}")
    print(f"Pages failed               : {stats.get('pages_failed', 0)}")
    print(f"PDFs downloaded            : {stats.get('pdfs_downloaded', 0)}")
    print(f"Office docs downloaded     : {stats.get('office_downloaded', 0)}")
    print(f"Images downloaded          : {stats.get('images_downloaded', 0)}")
    print(f"JS-rendering fallbacks used: {stats.get('js_fallback_used', 0)}")

    if not master_df.empty:
        print("\nDocuments by category:")
        print(master_df["category"].value_counts().to_string())

        print("\nDocuments by domain:")
        print(master_df["domain"].value_counts().to_string())

        print("\nDocuments by document type:")
        print(master_df["doc_type"].value_counts().to_string())

    def _dir_size_mb(path: Path) -> float:
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024 * 1024)

    print("\nDisk usage:")
    for label, path in [
        ("HTML snapshots", config.html_dir),
        ("PDFs", config.pdf_dir),
        ("Office Docs", config.office_dir),
        ("Images", config.images_dir),
        ("JSON documents", config.json_dir),
        ("Metadata", config.metadata_dir),
    ]:
        print(f"  {label:16s}: {_dir_size_mb(path):8.2f} MB")

    print("\n" + "=" * 70)
    print("Data collection stage complete. The dataset in `project/data/raw/`")
    print("is ready to be consumed by downstream parsing/OCR/chunking tasks.")
    print("=" * 70)


print_final_report(CONFIG, crawler.url_manager, run_stats, master_df)


MIT DATA COLLECTION — FINAL REPORT

Run window: 2026-07-27T19:41:26.723378+00:00  ->  2026-07-27T20:23:14.074656+00:00

Pages crawled successfully : 1000
Pages failed               : 17
PDFs downloaded            : 141
Office docs downloaded     : 2
Images downloaded          : 127
JS-rendering fallbacks used: 0

Documents by category:
category
academic_calendar          318
student_services           270
course_catalog             209
academic_records            90
academic_policies           75
subject_description         71
graduation_requirements     41
degree_requirements         21
admissions_information       9
graduate_education           3
campus_resources             2
student_handbook             1

Documents by domain:
domain
registrar.mit.edu      426
studentlife.mit.edu    353
catalog.mit.edu        323
oge.mit.edu              8

Documents by document type:
doc_type
html      845
pdf       136
image     127
office      2

Disk usage:
  HTML snapshots  :     4.73 MB
  PDF